In [ ]:
from plot import *
import os
folder = "2026_07_12" # for plotting
log_folder = "/Users/calebju/Code/hddp/logs/"

# Plot non-hierarchical experiments

This notebook goes over plotting dual gaps and out-of-sample performance for non-hierarchical problems.

## Gap bounds (Inventory, hydro-thermal, and risk-adverse inventory)

You need to adjust `xlim` and `ylim` to your liking.

In [ ]:
def plot_updated_gap(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname, is_huge_inv=False):
    
    lmbda = 0.9906 if large_lam else 0.8
    i_0 = 0 if large_lam else 24
    if is_huge_inv:
        i_0 = 0
        lmbda = 0.8 
    
    color_arr = ['red', 'black', 'blue', 'green', 'orange', 'purple']
    plt.style.use('ggplot')
    _, ax = plt.subplots(figsize=(5, 4))
    print(log_folder)
    experiment_log_folder = os.path.join(log_folder, "%s/exp_0" % date)
    name_arr = ["EDDP", "Inf-EDDP", "CE-Inf-EDDP", "Gap-Inf-EDDP", "CE-Inf-SDDP", "Cyc-SDDP"]
    idx_arr = [i_0+13] + [i_0+0, i_0+1, i_0+2] + list(range(i_0+3,i_0+13)) + list(range(i_0+14, i_0+24))
    folder_arr = [os.path.join(experiment_log_folder, "run_%d" % i) for i in idx_arr]
    bounds_arr, elpsed_times_arr, size_arr = read_logs(folder_arr)
    
    rng = np.random.default_rng(0)
    for i in range(4):
        n_iters = int(size_arr[i])
        xs = np.arange(n_iters)
    
        cutoff=-1
        if i==3 and env_name == "huge_inventory":
            cutoff=-3
        
        # apply monotone for upper bound
        np.minimum.accumulate(bounds_arr[i,:n_iters,1], out=bounds_arr[i,:n_iters,1])
        ax.plot(xs[:cutoff], bounds_arr[i,:n_iters+cutoff,0], linestyle=lss_arr[i], label=name_arr[i], color=color_arr[i])
        ax.plot(xs[:cutoff], bounds_arr[i,:n_iters+cutoff,1], linestyle=lss_arr[i], color=color_arr[i])
        
    # create and plot quantiles for Inf-SDDP (there is off-by-one error, so drop last iterate for SDDP)
    sddp_j_0_arr = [3, 14] # where the randon experiments start
    sddp_color_idx_arr = [4,5]
    for k,j_0 in zip(sddp_color_idx_arr,sddp_j_0_arr):
        n_iters = int(size_arr[j_0])-1
        xs = np.arange(n_iters)
        
        cutoff = -1
        if k==5:
            # P-SDDP has some weird off by three errors, so cut off last three
            cutoff=-5
        if k==4 and env_name == "huge_inventory":
            cutoff=-5
        
        inf_sddp_lb_arr = np.sort(bounds_arr[j_0:j_0+10,:n_iters,0], axis=0)
        inf_sddp_ub_arr = np.sort(bounds_arr[j_0:j_0+10,:n_iters,1], axis=0)
        # apply monotone for upper bound
        np.minimum.accumulate(inf_sddp_ub_arr, out=inf_sddp_ub_arr, axis=1)
        
        # median (average of 5 and 6th best)
        ax.plot(xs[:cutoff], np.mean(inf_sddp_lb_arr[4:6,:cutoff], axis=0), linestyle=lss_arr[k+4], label=name_arr[k], color=color_arr[k])
        ax.plot(xs[:cutoff], np.mean(inf_sddp_ub_arr[4:6,:cutoff], axis=0), linestyle=lss_arr[k+4], color=color_arr[k])
        
        # 10% and 90% quantile
        ax.fill_between(xs[:cutoff], inf_sddp_lb_arr[1,:cutoff], inf_sddp_lb_arr[8,:cutoff], color=color_arr[k], alpha=0.1)
        ax.fill_between(xs[:cutoff], inf_sddp_ub_arr[1,:cutoff], inf_sddp_ub_arr[8,:cutoff], color=color_arr[k], alpha=0.1)
        
    # ax.legend(loc=1)
    if not large_lam:
        ax.legend()
    ax.set(
        xlim=xlim,
        ylim=ylim,
        yscale='log', 
        xlabel="Iteration",
        ylabel="Optimal value bounds",
        title="Upper and lower bound gaps\n" + r"for {} and $\lambda={}$".format(env_name_in_title, lmbda),
    )
    plt.tight_layout()
                       
    if fname is None:
        plt.show()
    else:
        plt.savefig(fname, dpi=144)

### Inventory small

In [ ]:
env_name = "inventory"
env_name_in_title = "inventory (n=1)"
date = "2025_01_16"
large_lam=False
xlim, ylim = (-1,50), (1e1, 1e5)
fname = os.path.join(folder, '%s_gap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

In [ ]:
env_name = "inventory"
env_name_in_title = "inventory (n=1)"
date = "2025_01_16"
large_lam=True
xlim, ylim = (-10,1_000), (1e1, 1e7)
fname = os.path.join(folder, '%s_gap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

### Inventory (large)

In [ ]:
env_name = "inventory_large"
env_name_in_title = "inventory (n=10)"
date = "2026_07_12"
large_lam=False
xlim, ylim = (-4,300), (1e2,1e7)
fname = os.path.join(folder, '%s_gap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

In [ ]:
env_name = "inventory_large"
env_name_in_title = "inventory (n=10)"
date = "2026_07_12"
large_lam=True
xlim, ylim = (-10, 1_000), (1e2,1e9)
fname = os.path.join(folder, '%s_gap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

### Huge inventory

In [ ]:
env_name = "inventory_huge"
env_name_in_title = "inventory (n=32)"
date = "2026_07_13"
large_lam=False
xlim, ylim = (-4,375), (1e3,5e9)
fname = os.path.join(folder, '%s_gap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname, is_huge_inv=True)

### Risk-adverse

Note: EDDP (run_37) failed due to invalid `Pi`. We just copied the result from run_36 (`INF-SDDP`).

In [ ]:
env_name = "inventory-risk"
env_name_in_title = "risk-adverse inventory"
date = "2025_04_23"
large_lam=False
xlim, ylim = (-1,100), (1e3, 1e8)
fname = os.path.join(folder, '%s_gap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

### Hydro-thermal

In [ ]:
env_name = "hydro"
env_name_in_title = "hydro-thermal"
date = "2025_01_15"
large_lam=False
xlim, ylim = (-40,2_000), (1e6,5e8)
fname = os.path.join(folder, '%s_gap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

In [ ]:
env_name = "hydro"
env_name_in_title = "hydro-thermal"
date = "2025_01_15"
large_lam=True
xlim, ylim = (-40,2_000), (2e7,1e11)
fname = os.path.join(folder, '%s_gap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

## Optimal value vs runtime

In [ ]:
def plot_updated_gap_runtime(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname, is_huge_inv=False):
    
    lmbda = 0.9906 if large_lam else 0.8
    i_0 = 0 if large_lam else 24
    if is_huge_inv:
        i_0 = 0
        lmbda = 0.8 
    
    color_arr = ['red', 'black', 'blue', 'green', 'orange', 'purple']
    plt.style.use('ggplot')
    _, ax = plt.subplots(figsize=(5, 4))
    experiment_log_folder = os.path.join(log_folder, "%s/exp_0" % date)
    name_arr = ["EDDP", "Inf-EDDP", "CE-Inf-EDDP", "Gap-Inf-EDDP", "CE-Inf-SDDP", "Cyc-SDDP"]
    idx_arr = [i_0+13] + [i_0+0, i_0+1, i_0+2] + list(range(i_0+3,i_0+13)) + list(range(i_0+14, i_0+24))
    folder_arr = [os.path.join(experiment_log_folder, "run_%d" % i) for i in idx_arr]
    bounds_arr, elpsed_times_arr, size_arr = read_logs(folder_arr)
    
    rng = np.random.default_rng(0)
    for i in range(4):
        n_iters = int(size_arr[i])
        xs = elpsed_times_arr[i,:,0]
    
        cutoff=-1
        if i==3 and env_name == "huge_inventory":
            cutoff=-3
        
        # apply monotone for upper bound
        np.minimum.accumulate(bounds_arr[i,:n_iters,1], out=bounds_arr[i,:n_iters,1])
        ax.plot(xs[:n_iters+cutoff], bounds_arr[i,:n_iters+cutoff,0], linestyle=lss_arr[i], label=name_arr[i], color=color_arr[i])
        ax.plot(xs[:n_iters+cutoff], bounds_arr[i,:n_iters+cutoff,1], linestyle=lss_arr[i], color=color_arr[i])
        
    # create and plot quantiles for Inf-SDDP (there is off-by-one error, so drop last iterate for SDDP)
    sddp_j_0_arr = [3, 14] # where the randon experiments start
    sddp_color_idx_arr = [4,5]
    for k,j_0 in zip(sddp_color_idx_arr,sddp_j_0_arr):
        n_iters = int(size_arr[j_0])-1
        xs = np.mean(elpsed_times_arr[j_0:j_0+10,:,0], axis=0)
        
        cutoff = -1
        if k==5:
            # P-SDDP has some weird off by three errors, so cut off last three
            cutoff=-5
        if k==4 and env_name == "huge_inventory":
            cutoff=-5
        
        inf_sddp_lb_arr = np.sort(bounds_arr[j_0:j_0+10,:n_iters,0], axis=0)
        inf_sddp_ub_arr = np.sort(bounds_arr[j_0:j_0+10,:n_iters,1], axis=0)
        # apply monotone for upper bound
        np.minimum.accumulate(inf_sddp_ub_arr, out=inf_sddp_ub_arr, axis=1)
        
        # median (average of 5 and 6th best)
        ax.plot(xs[:n_iters+cutoff], np.mean(inf_sddp_lb_arr[4:6,:cutoff], axis=0), linestyle=lss_arr[k+4], label=name_arr[k], color=color_arr[k])
        ax.plot(xs[:n_iters+cutoff], np.mean(inf_sddp_ub_arr[4:6,:cutoff], axis=0), linestyle=lss_arr[k+4], color=color_arr[k])
        
        # 10% and 90% quantile
        ax.fill_between(xs[:n_iters+cutoff], inf_sddp_lb_arr[1,:cutoff], inf_sddp_lb_arr[8,:cutoff], color=color_arr[k], alpha=0.1)
        ax.fill_between(xs[:n_iters+cutoff], inf_sddp_ub_arr[1,:cutoff], inf_sddp_ub_arr[8,:cutoff], color=color_arr[k], alpha=0.1)
        
    # ax.legend(loc=1)
    if not large_lam:
        ax.legend()
    ax.set(
        xlim=xlim,
        ylim=ylim,
        yscale='log', 
        xlabel="Runtime (sec)",
        ylabel="Optimal value bounds",
        title="Upper and lower bound gaps\n" + r"for {} and $\lambda={}$".format(env_name_in_title, lmbda),
    )
    plt.tight_layout()
                       
    if fname is None:
        plt.show()
    else:
        plt.savefig(fname, dpi=144)

### Inventory runtime comparison

In [ ]:
env_name = "inventory"
env_name_in_title = "inventory (n=1)"
date = "2025_01_16"
large_lam=False
xlim, ylim = (-1,60), (1e1, 1e5)
fname = os.path.join(folder, '%s_runtimegap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap_runtime(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

In [ ]:
env_name = "inventory"
env_name_in_title = "inventory (n=1)"
date = "2025_01_16"
large_lam=True
xlim, ylim = (-10,1_200), (1e1, 1e7) 
fname = os.path.join(folder, '%s_runtimegap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap_runtime(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

### Inventory (large) runtime comparison 

In [ ]:
env_name = "inventory_large"
env_name_in_title = "inventory (n=10)"
date = "2026_07_12"
large_lam=False
xlim, ylim = (-4,1_800), (1e2,1e7)
fname = os.path.join(folder, '%s_gap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap_runtime(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

### Hydro-thermal runtime comparison

In [ ]:
env_name = "hydro"
env_name_in_title = "hydro-thermal"
date = "2025_01_15"
large_lam=False
xlim, ylim = (-40,1_200), (1e6,5e8)
fname = os.path.join(folder, '%s_runtimegap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap_runtime(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

In [ ]:
env_name = "hydro"
env_name_in_title = "hydro-thermal"
date = "2025_01_15"
large_lam=True
xlim, ylim = (-40,3_600), (2e7,1e11)
fname = os.path.join(folder, '%s_runtimegap_gamma=%s.pdf' % (env_name, "0.99" if large_lam else "0.8"))
plot_updated_gap_runtime(env_name, env_name_in_title, date, large_lam, xlim, ylim, fname)

In [ ]:
### Risk-adverse inventory runtime comparison

In [ ]:
# TODO

In [ ]:
# TODO

## Average runtime (Inventory, hydro-thermal, and risk-adverse inventory)

We will remove `eval_time` from the total time since this is to track the progress.

In [ ]:
def get_average_runtime(env_name, date, large_lam, is_huge_inv=False):
    lmbda = 0.9906 if large_lam else 0.8
    i_0 = 0 if large_lam else 24
    if is_huge_inv:
        i_0 = 0
        lmbda = 0.8 
        
    experiment_log_folder = os.path.join(log_folder, "%s/exp_0" % date)
    idx_arr = [i_0+13] + [i_0+0, i_0+1, i_0+2] + list(range(i_0+3,i_0+13)) + list(range(i_0+14, i_0+24))
    folder_arr = [os.path.join(experiment_log_folder, "run_%d" % i) for i in idx_arr]
    bounds_arr, elpsed_times_arr, size_arr = read_logs(folder_arr)
    env_lambda_name = "%s_lambda=%s" % (env_name, lmbda)
    name_arr = ["EDDP", "Inf-EDDP", "CE-Inf-EDDP", "Gap-Inf-EDDP", "CE-Inf-SDDP", "Cyc-SDDP"]

    print("env_lambda & " + " & ".join(name_arr))
    msg = "%s_lambda=%s " % (env_name, lmbda)
    for i in range(4):
        # subtract cutoff for safe measure
        n_iters = int(size_arr[i])-5 
        # total_time,fwd_time,select_time
        avg_total_time = elpsed_times_arr[i, n_iters-1,0]/n_iters
        avg_fwd_time   = elpsed_times_arr[i, n_iters-1,1]/n_iters
        avg_select_time= elpsed_times_arr[i, n_iters-1,2]/n_iters
        avg_eval_time  = elpsed_times_arr[i, n_iters-1,3]/n_iters
        
        msg += "& %.2e " % (avg_total_time-avg_eval_time)
        
    sddp_j_0_arr = [3, 14] # where the randon experiments start
    sddp_color_idx_arr = [4,5]
    for j,(k,j_0) in enumerate(zip(sddp_color_idx_arr,sddp_j_0_arr)):
        n_iters = int(size_arr[j_0])-5
        # total_time,fwd_time,select_time
        avg_total_time = np.mean(elpsed_times_arr[j_0:j_0+10,n_iters-1,0], axis=0)/n_iters
        avg_fwd_time   = np.mean(elpsed_times_arr[j_0:j_0+10,n_iters-1,1], axis=0)/n_iters
        avg_select_time= np.mean(elpsed_times_arr[j_0:j_0+10,n_iters-1,2], axis=0)/n_iters
        avg_eval_time  = np.mean(elpsed_times_arr[j_0:j_0+10,n_iters-1,3], axis=0)/n_iters
        
        msg += "& %.2e " % (avg_total_time-avg_eval_time)
    
    return msg

### Inventory per-iteration runtime

In [ ]:
env_name = "inventory"
date = "2025_01_16"
large_lam=False
print(get_average_runtime(env_name, date, large_lam))

In [ ]:
env_name = "inventory"
date = "2025_01_16"
large_lam=True
print(get_average_runtime(env_name, date, large_lam))

### Inventory (large) per-iteration runtime

In [ ]:
env_name = "inventory"
date = "2026_07_12"
large_lam=False
print(get_average_runtime(env_name, date, large_lam))

In [ ]:
env_name = "inventory"
date = "2026_07_12"
large_lam=True
print(get_average_runtime(env_name, date, large_lam))

### Inventory (huge) per-iteration runtime

In [ ]:
env_name = "inventory"
date = "2026_07_13"
large_lam=False
print(get_average_runtime(env_name, date, large_lam, is_huge_inv=True))

### Hydro per-iteration runtime

In [ ]:
env_name = "hydro"
date = "2025_01_15"
large_lam=False
print(get_average_runtime(env_name, date, large_lam))

In [ ]:
env_name = "hydro"
date = "2025_01_15"
large_lam=True
print(get_average_runtime(env_name, date, large_lam))

### Risk-adverse inventoroy per-iteration runtime

In [ ]:
# TODO

In [ ]:
# TODO

## Upper bound model time comparison (before and after amoritzation)

In our previous, the upper bound had only a single Gurobi model for the upper bound model.

However, this required updating the RHS for each different scenario.
In our amoritized code, we created multiple Gurobi models, where each was dedicated to a different scenario to prevent over-writing.

In [ ]:
def runtime_breakdown(env_name, date, large_lam, is_huge_inv=False):
    lmbda = 0.9906 if large_lam else 0.8
    i_0 = 0 if large_lam else 24
    if is_huge_inv:
        i_0 = 0
        lmbda = 0.8 
        
    experiment_log_folder = os.path.join(log_folder, "%s/exp_0" % date)
    idx_arr = [i_0+2]
    folder_arr = [os.path.join(experiment_log_folder, "run_%d" % i) for i in idx_arr]
    bounds_arr, elpsed_times_arr, size_arr = read_logs(folder_arr)

    n_iters = int(size_arr[0])-5 
    # we remove comm time (after eval time)
    avg_elpsed_times = np.append(elpsed_times_arr[0,n_iters-1,0:4,], elpsed_times_arr[0,n_iters-1,5:])/n_iters

    # elpsed_times_arr keys: 
    time_key_arr = ["Tot", "Fwd", "Select", "Eval", "Gap (lb-up)", "Gap (ub-up)", "Gap (ub-solve)"]

    print("env_lambda & " + " & ".join(time_key_arr))
    msg = "%s_lambda=%s " % (env_name, lmbda)
    for t in avg_elpsed_times:
        msg += " & %.2e " % t
    msg += "\npercentage "
    for t in avg_elpsed_times:
        msg += " & %.3f " % (t/avg_elpsed_times[0])
    return msg

### Inventory (amoritized update)

In [ ]:
env_name = "inventory"
date = "2025_01_16"
large_lam=True
print(runtime_breakdown(env_name, date, large_lam))

### Inventory (non-amoritized)

In [ ]:
env_name = "inventory"
date = "2025_01_16_legacy"
large_lam=True
print(runtime_breakdown(env_name, date, large_lam))